In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

evaluation_path = Path("../results/evaluation/evaluation_results.csv")

if not evaluation_path.exists():
    evaluation_path = Path("/mnt/data/evaluation_results.csv")

df = pd.read_csv(evaluation_path)

required_columns = [
    "system",
    "query_id",
    "query",
    "rank",
    "source_row_id",
    "title",
    "similarity_score",
    "synopsis",
    "relevance"
]

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if len(df) != 60:
    raise ValueError(f"Expected 60 evaluation rows, found {len(df)}")

print("Evaluation rows:", len(df))
print("Queries:", df["query_id"].nunique())
print("Systems:", df["system"].unique().tolist())

Evaluation rows: 60
Queries: 6
Systems: ['TF-IDF', 'Transformer']


In [2]:
case_labels = {
    3: "Transformer clearly succeeds over TF-IDF",
    2: "TF-IDF competitive / slightly better ranking",
    6: "Both systems succeed",
    5: "Both systems struggle",
    4: "Major ranking difference"
}

case_summary = []

for query_id, label in case_labels.items():
    query_rows = df[df["query_id"] == query_id]
    query_text = query_rows["query"].iloc[0]
    metrics = []

    for system in ["TF-IDF", "Transformer"]:
        system_rows = query_rows[query_rows["system"] == system].sort_values("rank")
        relevance = system_rows["relevance"].tolist()
        precision = np.mean([value > 0 for value in relevance])
        metrics.append(
            {
                "system": system,
                "precision_at_5": precision,
                "mean_relevance": np.mean(relevance)
            }
        )

    case_summary.append(
        {
            "query_id": query_id,
            "case": label,
            "query": query_text,
            "tfidf_p5": metrics[0]["precision_at_5"],
            "transformer_p5": metrics[1]["precision_at_5"],
            "tfidf_mean_relevance": metrics[0]["mean_relevance"],
            "transformer_mean_relevance": metrics[1]["mean_relevance"]
        }
    )

case_summary_df = pd.DataFrame(case_summary)
case_summary_df

,query_id,case,query,tfidf_p5,transformer_p5,tfidf_mean_relevance,transformer_mean_relevance
0,3,Transformer clearly succeeds over TF-IDF,കുറ്റകൃത്യങ്ങളും ത്രില്ലർ കഥയും,0.2,0.6,0.4,1.0
1,2,TF-IDF competitive / slightly better ranking,പ്രണയവും പ്രണയ ബന്ധങ്ങളും,0.8,0.8,1.6,1.4
2,6,Both systems succeed,സാമൂഹിക പ്രശ്നങ്ങളും കുടുംബ നാടകവും,1.0,1.0,1.4,1.6
3,5,Both systems struggle,ആക്ഷനും പ്രതികാരവും,0.6,0.4,0.8,0.6
4,4,Major ranking difference,സൗഹൃദവും സുഹൃത്തുക്കളുടെ ബന്ധവും,0.6,0.8,1.0,1.2


In [3]:
def show_case(query_id):
    query_rows = df[df["query_id"] == query_id].sort_values(["system", "rank"])
    query_text = query_rows["query"].iloc[0]

    print("=" * 100)
    print(f"Query {query_id}: {query_text}")
    print("=" * 100)

    for system in ["TF-IDF", "Transformer"]:
        system_rows = query_rows[query_rows["system"] == system].sort_values("rank")
        print()
        print(system)
        print("-" * 100)

        for _, row in system_rows.iterrows():
            print(f"Rank: {int(row['rank'])}")
            print(f"Title: {row['title']}")
            print(f"Similarity score: {row['similarity_score']}")
            print(f"Relevance: {int(row['relevance'])}")
            print(f"Synopsis: {row['synopsis']}")
            print()

In [4]:
show_case(3)

Query 3: കുറ്റകൃത്യങ്ങളും ത്രില്ലർ കഥയും

TF-IDF
----------------------------------------------------------------------------------------------------
Rank: 1
Title: James and Alice
Similarity score: 0.0701616738600166
Relevance: 0
Synopsis: അയാളും ഞാനും തമ്മിലിന്റെ അഭിനയമികവിലും എന്ന് നിന്റെ മൊയ്തീന്റെ ജനപ്രിയതയിലും അമർ അക്ബർ അന്തോണിയുടെ ഓളപരപ്പിലും പ്രേക്ഷകരുടെ രാജുവേട്ടനായി മാറിയ പൃഥ്വിരാജിന്റെ, രാജപ്പനെന്നു വിളിപ്പിച്ച പഴയ വെറുപ്പിക്കൽ മോഡിലെക്കുള്ള തിരിച്ചുപോക്കിന്റെ പൂർണതയാണ് ജെയിംസ് ആൻഡ് ആലീസ്. അടുത്ത കാലത്തെ പൃഥ്വിരാജ് സിനിമകളുടെ ( മെമ്മറീസ്, സെവന്‍ത് ഡേ, അമർ അക്ബർ അന്തോണി, അനാർക്കലി ) ആസ്ഥാന ഛായാഗ്രാഹകൻ സുജിത് വാസുദേവിന്റെ ആദ്യ സംവിധാന സംരംഭമാണിത്. അതിനേക്കാൾ സിറ്റി ഓഫ് ഗോഡിനും ദൃശ്യത്തിനും ക്യാമറ ചെയ്തൊരാൾ എടുക്കുന്ന ആദ്യ സിനിമക്കെങ്ങനെ ഇത്ര കൂതറ സ്ക്രിപ്റ്റുണ്ടായെന്നതാണൽഭുതം.പൃഥ്വിരാജിന്റെ മാർക്കറ്റ് വാല്യുവിൽ രക്ഷപെടുമെന്ന പ്രതീക്ഷയിലായിരിക്കും. ഫൈൻ ആർട്സ് ഗ്രാജുവേറ്റ് ജെയിംസ്, പ്ലാന്ററുടെ മകൾ ആലീസിനെ പ്രണയിച്ചു വിവാഹം കഴിച്ച ശേഷമുള്ള അസ്വാരസ്യങ്ങളാണ് ജെയിംസ് ആൻഡ് ആലീസിന്റെ 

In [5]:
show_case(2)

Query 2: പ്രണയവും പ്രണയ ബന്ധങ്ങളും

TF-IDF
----------------------------------------------------------------------------------------------------
Rank: 1
Title: Hello Namasthe
Similarity score: 0.0860714011187624
Relevance: 2
Synopsis: കല്യാണപെണ്ണുമായി വിവാഹത്തലേന്നു ഒളിച്ചോടുന്ന ആർ ജെ ജെറി (സഞ്ജു ശിവറാം) അഭയം പ്രാപിക്കുന്നത് സുഹൃത്തും കൊളീഗുമായ ആർ ജെ മാധവിന്റെ ( വിനയ് ഫോർട്ട്‌ ) വീട്ടിലാണ്. അങ്ങനെ ജെറിയും അന്നയും (മിയ), മാധവും പ്രിയയും (ഭാവന), ഒരു മനസ്സോടെ ഇരു കുടുംബങ്ങളായി ജീവിക്കുന്നു. സംയുക്തമായി സിറ്റിയിലെ ഒരു പോഷ് ഏരിയയിൽ വില്ല വാങ്ങി സന്തുഷ്ട ജീവിതം തുടരവേ ഒരു ചെറിയ സംഭവം അടുപ്പത്തിൽ വിള്ളലുണ്ടാക്കുന്നു. ഒരാളുടെ മതിലതിർത്തിക്കുള്ളിൽ നിൽക്കുന്ന പ്ലാവിൽ നിന്ന് വീഴുന്ന ചക്കയും ചവറിലകളും മറ്റൊരാൾക്കു പ്രശ്നമാകുന്നതും, ഈ രണ്ടു കുടുംബങ്ങളുടെയും വഴക്ക് മൊത്തം കോളനിക്ക് തന്നെ തലവേദനയാകുന്നതുമാണ് ബാക്കി കഥ. പഴയകാല സിദ്ദിക്ക് ജയറാം റിച് കോളനി ലൈഫ് സിനിമകളുടെ റൂട്ടിലാണ്‌ കൃഷ്ണൻ പൂജപ്പുരയുടെ തിരക്കഥ. ഇതേ തിരക്കഥ കൃഷ്ണൻ പൂജപ്പുരയുടെ സ്ഥിരം സംവിധായകൻ സജി സുരേന്ദ്രന്റെ കയ്യിൽ കിട്ടിയാലുള്ളത്ര വി

In [6]:
show_case(6)

Query 6: സാമൂഹിക പ്രശ്നങ്ങളും കുടുംബ നാടകവും

TF-IDF
----------------------------------------------------------------------------------------------------
Rank: 1
Title: Monsoon Mangoes
Similarity score: 0.0992571954853926
Relevance: 1
Synopsis: ജനപ്രിയ ടെലിവിഷൻ പരമ്പര അക്കരകാഴ്ചകളുടെ സംവിധായകന്‍റെ ആദ്യ സിനിമയെന്ന പേരിലാണ് മൺസൂൺ മാംഗോസ് അനൗൻസ് ചെയ്യപെട്ടത്‌. പിന്നീടത്‌ മഹേഷിന്‍റെ പ്രതികാരത്തിനും മുൻപേ ഫഹദിന്‍റെ തിരിച്ചുവരവ് ചിത്രമെന്ന നിലയിലായി. വൃത്തിയുള്ള പോസ്റ്ററും ഡിസ്ക്കോ സോങ്ങ് പ്രോമോയും അതിശയിപ്പിക്കുന്ന ഷോട്ടുകൾ നിറഞ്ഞ ട്രെയിലറുമെത്തിയപ്പോൾ ഈ വർഷത്തെ നല്ല മലയാള സിനിമയെന്ന തോന്നലുണ്ടാക്കി. ഒടുവിൽ അണ്ടിയോടടുക്കുമ്പോൾ ജനം മാങ്ങയുടെ പുളിയറിഞ്ഞു. കാരണം, കാർബൈഡ് ഇട്ടു പഴുപ്പിച്ചതാണീ മൺസൂൺ മാംഗോസ്. തൊലിപ്പുറത്തുള്ള ഭംഗിയേയുള്ളൂ ! ഫഹദ് ഇത്തവണയും വിസ്മയിപ്പിക്കുന്നുണ്ട്. പക്ഷെ വിവാദ ഇടവേളയ്ക്കു ശേഷമുള്ള തിരിച്ചു വരവിനുതകുന്ന റീലോഞ്ച് വെഹിക്കിൾ ആയിരുന്നില്ല മൺസൂൺ മാംഗോസെന്നുമാത്രം. മികച്ച പ്രൊഡക്ഷൻ ക്വാളിറ്റിയും ആക്ടേർസിന്‍റെ പെര്‍ഫോമെന്‍സുമാണ് സിനിമയുടെ പ്ലസ്‌ പോയന്‍റ്. എന്നാൽ തെളിമയുള്ള

In [7]:
show_case(5)

Query 5: ആക്ഷനും പ്രതികാരവും

TF-IDF
----------------------------------------------------------------------------------------------------
Rank: 1
Title: Kismath
Similarity score: 0.0611381257392071
Relevance: 0
Synopsis: രാജീവ് രവിയുടെ 'അന്നയും റസൂലി'ന്റെ ആഴവും പരപ്പും കുറഞ്ഞ തിരിച്ചിടലാണ് രാജീവ് രവി തന്നെ നി‍ർമാതാക്കളിലൊരാളാകുന്ന നവാഗതനായ ഷാനവാസ് ബാവക്കുട്ടിയുടെ 'കിസ്മത്ത്'. വ്യത്യസ്ത മതങ്ങളിൽ/ജാതികളിൽ പെട്ട ഇരുഹൃദയങ്ങൾ പ്രണയിക്കുമ്പോളുണ്ടാകുന്ന സിസ്റ്റത്തിന്റെ പുകച്ചിലുകൾ. ഇക്കഥയിലെ വലിയ സദാചാരപോലീസ് സമൂഹത്തെക്കാൾ വലിയ പോലീസാകുന്ന പോലീസാണെന്നു മാത്രം. മഹല്ല് അധികാരിയുടെ മകനായ മുസ്ലിം യുവാവും പ്രായക്കൂടുതലുള്ള ദളിത് യുവതിയും. ജാതിയുടെയും മതത്തിന്റെയും സാമൂഹികസമവാക്യങ്ങൾ തുറന്നു സംസാരിക്കാൻ ശ്രമിക്കുന്ന തരക്കേടില്ലാത്തൊരു പ്രണയകാലം ആണ് കിസ്മത്ത്. അക്രമത്തെക്കാൾ വലിയ അനീതി നിസ്സംഗതയാണെന്നുള്ള പ്രവാചക വചനത്തോടെയാണ് സിനിമ ആരംഭിക്കുന്നത്. പൊന്നാനിയുടെ ട്രൂ സ്റ്റോറി കൂടിയാണീ കഥ. പ്രണയനായകൻ ഇർഫാനായി നടൻ അഭിയുടെ മകൻ ഷെയിനും അനിതയായി മോഡൽ ശ്രുതി മേനോനും പെർഫെക്ട് കാസ്റ്റിംഗ് ഉം കെമിസ്ട്രിയുമാണ

In [8]:
show_case(4)

Query 4: സൗഹൃദവും സുഹൃത്തുക്കളുടെ ബന്ധവും

TF-IDF
----------------------------------------------------------------------------------------------------
Rank: 1
Title: Yezhde
Similarity score: 0.0525139805448985
Relevance: 1
Synopsis: കൗതുകമുണർതുന്ന ടൈറ്റിൽ. ആകർഷണീയമായ ഫോണ്ട് ഡിസൈൻ. രസകരമായ ട്രെയിലർ. റിലീസിന് മുൻപേ ഇഷ്ടം പിടിച്ചു പറ്റിയ യെസ്ഡി പക്ഷെ പ്രധാന റെയ്സിൽ നിരാശപെടുത്തുകയാണ്. നവാഗതനായ അരുൺ ഓമന സദാനന്ദന്റെ മാനസാന്തരപെട്ട യെസ്ഡി 'ആമേനി'നും 'മറിയംമുക്കി'നും നടുവിലൂടെ ഓടുന്നൊരു കോമിക് ഫാന്ടസിയാണ്. അതും ഒരു ഇന്റലെക്ച്വൽ ചുവയോടെ. എന്പത്‌കളിലെക്കൊക്കെ ഓടിയെതുന്നൊരു നാടൻ പിരീഡ് സിനിമ കൂടിയാണ്. വർക്ക് ഷോപ്പ് ഓണർ പഴയകാല ഫ്രീക്കൻ ചെത്തനാപ്പിയും ( പി ബാലചന്ദ്രന്റെ മാരകറോൾ) സഹോദരൻ മെക്കാനിക് പാപ്പിയും ( ജയൻ ചേർത്തല) തമ്മിലുള്ള വഴക്ക് തീർക്കുന്നത് നാട്ടിൻപുറത്തെ ബൈക്ക് റെസിലൂടെയാണ്. ചെത്തനാപ്പിയുടെ ചെതാനെങ്കിൽ എഴുപതിയെട്ടിൽ ബോംബയിൽ നിന്ന് വാങ്ങിയൊരു കിടിലൻ യെസ്ഡി വെച്ചും. ഈ യെസ്ഡി മോഷ്ടിച്ച ഫീലിപ്പൊസിൽ (ഇന്ദ്രൻസ്) നിന്നാണ് സിനിമ തുടങ്ങുന്നത്. ചെത്തനാപ്പിയുടെ ലോക്കൽ ബൈക്ക് ഗാങ്ങിന്റെ ആരാധനാമൂർത

In [9]:
overlap_rows = []

for query_id in case_labels:
    query_rows = df[df["query_id"] == query_id]

    tfidf_titles = set(
        query_rows[query_rows["system"] == "TF-IDF"]["title"]
    )

    transformer_titles = set(
        query_rows[query_rows["system"] == "Transformer"]["title"]
    )

    overlap = tfidf_titles.intersection(transformer_titles)

    overlap_rows.append(
        {
            "query_id": query_id,
            "tfidf_top5_count": len(tfidf_titles),
            "transformer_top5_count": len(transformer_titles),
            "shared_titles": len(overlap),
            "shared_titles_list": ", ".join(sorted(overlap))
        }
    )

pd.DataFrame(overlap_rows)

,query_id,tfidf_top5_count,transformer_top5_count,shared_titles,shared_titles_list
0,3,5,5,0,
1,2,5,5,0,
2,6,5,5,0,
3,5,5,5,1,Maheshinte Prathikaaram
4,4,5,5,0,
